# 1 — state, nodes, edges

This project builds a trip planner. You describe a trip — cities, dates,
budget, what you like — and it researches each city, picks hotels,
activities and trains, checks the total against your budget, shows you a
draft, and books only what you approve. Some of those steps need a model
(reading your description, choosing activities); others are plain
arithmetic (adding up the cost, checking the dates). The planner is built as
a **graph**: each step is a box, called a *node*, and arrows between the
boxes, called *edges*, say what runs next. Every node reads from and writes
to one shared record of the trip so far, called the *state*.

This book builds the first two steps, checking that a trip description is
complete and asking for what's missing, and the terminal app you'll use
to talk to the planner. There is no model anywhere in this book: you type
the trip in a fixed format, and the next book lets you type it in words.

In [ ]:
from tripgraph.config import configure_tracing
from tripgraph.domain.models import TripBrief
from tripgraph.domain.rules import PROBLEMS, missing_fields, nights, split_nights

configure_tracing(book=1)

**The brief.** A *brief* is what the traveller asked for: which cities, from
which date to which date, the budget for the whole trip, and what they're
interested in. Dates are written as ISO dates — year, month, day, so the
10th of October 2026 is `"2026-10-10"`. `end` is the day you leave the
last city, not the last night you sleep there.

Two briefs are used all through this book. `COMPLETE` has everything.
`NO_DATES` is the same trip with the dates left out — the kind of brief
someone types when they haven't decided yet. Every field of `TripBrief` has
a default value, so `NO_DATES` is still a perfectly good `TripBrief`: the
object doesn't refuse to exist just because the trip isn't fully described.

In [ ]:
COMPLETE = TripBrief(
    cities=["Lisbon", "Porto"],
    start="2026-10-10",
    end="2026-10-15",
    budget=900,
    interests=["food", "museums"],
)
NO_DATES = COMPLETE.model_copy(update={"start": None, "end": None})

print(COMPLETE)
print(NO_DATES)

**Rules are plain functions.** Whether a brief is complete, and how many
nights each city gets, are questions with exact answers, so they're
answered by ordinary Python functions in `tripgraph/domain/rules.py`, not
by a model.

`missing_fields(brief)` returns a list of problem names: `"start"` if there's
no start date, `"budget"` if there's no budget, and so on. An empty list means
the brief is complete. Each name is a key in `PROBLEMS`, which holds the
question to ask the traveller to fix it.

`nights(brief)` counts the nights between `start` and `end`. Before running:
how many nights is 2026-10-10 to 2026-10-15?

`split_nights(brief)` shares those nights across the cities, in the order the
brief lists them. Each city gets the whole-number share, and any nights left
over go one each to the first cities. As numbers, for 11 nights across 4
cities: `11 // 4` is 2, so every city gets 2; `11 % 4` is 3 nights left over,
so the first three cities get one more — 3, 3, 3, 2. It returns
`(city, check_in, check_out)` for each city, where each check-out is the next
city's check-in. Predict the split for `COMPLETE` (Lisbon and Porto), and for
the same trip with Madrid added and the end moved two days later.

In [ ]:
print("COMPLETE: ", missing_fields(COMPLETE))
print("NO_DATES: ", missing_fields(NO_DATES))
for problem in missing_fields(NO_DATES):
    print("   ask:", PROBLEMS[problem])
print()

print("nights:", nights(COMPLETE))
for stay in split_nights(COMPLETE):
    print(stay)
print()

THREE_CITIES = COMPLETE.model_copy(update={"cities": ["Lisbon", "Porto", "Madrid"], "end": "2026-10-17"})
print("nights:", nights(THREE_CITIES))
for stay in split_nights(THREE_CITIES):
    print(stay)

**Rules have tests.** Because the rules are plain functions, you can check
them without running anything else: no graph, no model, no API key. A
*test* is a function that calls a rule with an input you've worked out by
hand and states the answer it must give, with `assert`. `pytest` finds every
file named `test_*.py` under `tests/`, runs every function in it whose name
starts with `test_`, and reports the ones whose `assert` failed.

Below: one of the tests from `tests/domain/test_rules.py` — the split you
just predicted — and then the whole file run by `pytest`. Each dot in the
output is a test that passed.

In [ ]:
import sys

!sed -n '/def test_five_nights_two_cities/,/^$/p' tests/domain/test_rules.py
!{sys.executable} -m pytest tests/domain -q